# 01 · Landing
Descarga el zip del día desde NOAA y deja el CSV en `landing/csv`, la carpeta que vigila Auto Loader. Es idempotente: si el archivo ya existe no lo repite.

In [0]:
%run ./00_config

In [0]:
import os, re, shutil
from datetime import timedelta

dbutils.widgets.text("fecha", "auto")
fecha = dbutils.widgets.get("fecha")

# 1. Zips dejados en la carpeta de entrada
entrada = V("entrada")
pendientes = sorted(f for f in os.listdir(entrada) if re.fullmatch(r"AIS_\d{4}_\d{2}_\d{2}\.zip", f))
fechas = []
for z in pendientes:
    shutil.move(f"{entrada}/{z}", f"{V('raw_zips')}/{z}")
    y, m, d = z[4:14].split("_")
    fechas.append(f"{y}-{m}-{d}")
if pendientes:
    print("zips recibidos en entrada:", pendientes)

# 2. Sin archivos nuevos:
#    - "solo_entrada": job de sondeo → no hay nada que hacer
#    - "auto": job diario → siguiente día al último en plata (descarga de NOAA)
#    - "YYYY-MM-DD": ejecución manual de una fecha
if not fechas and fecha != "solo_entrada":
    if fecha == "auto":
        t = T("silver", "ais_posiciones")
        ult = spark.sql(f"SELECT max(fecha) FROM {t}").first()[0] if spark.catalog.tableExists(t) else None
        fecha = str(ult + timedelta(days=1)) if ult else "2023-06-01"
    fechas = [fecha]

hay_datos = "true" if fechas else "false"
fecha_lista = ",".join(fechas)
try:
    dbutils.jobs.taskValues.set(key="fecha", value=fecha_lista)
    dbutils.jobs.taskValues.set(key="hay_datos", value=hay_datos)
except Exception as e:
    print("taskValues no disponible en este contexto:", str(e)[:80])
print("hay_datos:", hay_datos, "| fechas a procesar:", fecha_lista or "(ninguna)")

In [0]:
import zipfile, urllib.request

csv_dir = V("csv")

for fecha in fechas:
    y, m, d = fecha.split("-")
    nombre = f"AIS_{y}_{m}_{d}"
    url = f"https://coast.noaa.gov/htdata/CMSP/AISDataHandler/{y}/{nombre}.zip"
    zip_path  = f"{V('raw_zips')}/{nombre}.zip"
    csv_final = f"{csv_dir}/{nombre}.csv"

    # 1. Descarga (se salta si el zip ya está, por ejemplo si llegó por la carpeta de entrada)
    if not os.path.exists(zip_path):
        print("descargando", url)
        urllib.request.urlretrieve(url, zip_path)
    print(nombre, "zip:", round(os.path.getsize(zip_path) / 1e6, 1), "MB")

    # 2. Descompresión a temporal y move a la carpeta vigilada por Auto Loader
    if not os.path.exists(csv_final):
        tmp_dir = f"{V('raw_zips')}/_tmp_{nombre}"
        os.makedirs(tmp_dir, exist_ok=True)
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(tmp_dir)
        csv_tmp = [os.path.join(r, f) for r, _, fs in os.walk(tmp_dir) for f in fs if f.endswith(".csv")][0]
        shutil.move(csv_tmp, csv_final)
        shutil.rmtree(tmp_dir)
    print(nombre, "csv:", round(os.path.getsize(csv_final) / 1e6, 1), "MB")